# **Langgraph Agents Evaluation - Websearch Tools, Token usages, Information Delivery**

### Rafael Galán López

## **Introduccion y Marco de Evaluacion**

El presente notebook (`notebooks/agent_evaluations/langgraph_agents_evaluation.ipynb`) tiene como objetivo realizar un analisis tecnico y comparativo sobre el rendimiento de los agentes basados en LangGraph desarrollados en el modulo `src/rafael_langgraph_websearchagent/agents/`.

---

### 1. Foco de Evaluacion Principal: `A002_langgraph_agent_refined.py`

En la primera parte de este estudio nos centramos en evaluar a fondo el agente Refined (`A002`), el cual representa la base de los modelos Refined Plus `A003` y Refined Plus v2 `A004`. Analizaremos el comportamiento de su herramienta de busqueda web (`TavilySearch`) bajo los siguientes ejes clave:

1. **Estructura y Payload de la `search_tool`:** Analisis de la informacion devuelta por Tavily, la API inyectada en el prompt inicial mediante `bind_tools()` y el coste en tokens del `ToolMessage`.
2. **Generacion de Snippets y Score:** Evaluacion del metodo de extraccion de texto, la precision del score de similitud semantica y la relevancia real de los fragmentos recuperados frente a la consulta del usuario.
3. **Fidelidad Informativa vs. Alucinacion:** Diagnostico sobre como el LLM procesa las observaciones devueltas, identificando riesgos de sintesis incompleta o relleno con conocimiento estatico.

---

### 2. Evolucion y Hoja de Ruta de Optimizacion

Tras diagnosticar las limitaciones de la version Refined (`A002`), el notebook aborda la transicion hacia las versiones mejoradas:

- **Agent Refined Plus v1** (`A003_langgraph_agent_refined_plus.py`): Evaluacion de la desinfeccion de payload, truncado de respuesta y el filtro semantico estricto con Pydantic (`relevance_grader`).
- **Agent Refined Plus v2** (`A004_langgraph_agent_refined_plusv2.py`): Analisis de la version final optimizada, demostrando la flexibilizacion del evaluador de relevancia y la aplicacion de un limite de iteraciones para maximizar la precision reduciendo drasticamente el consumo de tokens.

## **Configuracion del entorno e Importacion de modulos**

En esta celda inicial se cargan las dependencias del proyecto, las variables de entorno necesarias para autenticar las peticiones a los modelos de lenguaje (LLM) y al motor de busqueda web, asi como las herramientas y agentes desarrollados en el paquete `src.rafael_langgraph_websearchagent`.

#### Requisitos de API Keys en `.env`
Para la correcta ejecucion de las celdas de evaluacion, es necesario contar con un archivo `.env` en la raiz del proyecto que contenga las siguientes claves de API[cite: 7]:

* `ANTHROPIC_API_KEY`: Clave de acceso al modelo de lenguaje (Claude de Anthropic).
* `TAVILY_API_KEY`: Clave de acceso al motor de busqueda y extraccion web (Tavily).

In [ ]:
import sys
from dotenv import load_dotenv

# Añadir la raíz del proyecto al sys.path para importar correctamente el paquete local
sys.path.append("../../")

# Cargar API Keys de Anthropic y Tavily
load_dotenv()

# Importaciones de LangChain y LangGraph
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch, TavilyExtract
from langchain.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

# Importación de los agentes locales desde el paquete
from src.rafael_langgraph_websearchagent.agents.A002_langgraph_agent_refined import agent as agent_refined
from src.rafael_langgraph_websearchagent.agents.A003_langgraph_agent_refined_plus import agent as agent_plus_v1
from src.rafael_langgraph_websearchagent.agents.A004_langgraph_agent_refined_plusv2 import agent as agent_plus_v2

print("✅ Entorno e importaciones cargadas correctamente.")

## **1. Explicación del Agente A002 Refined y de la search_tool**

En esta sección se detalla la arquitectura técnica y el mecanismo de funcionamiento de la herramienta de búsqueda web (`TavilySearch`) integrada en el agente Refined (`A002_langgraph_agent_refined.py`). Se describe la estructura del payload devuelto por la herramienta, el criterio de decisión del modelo para invocar la API externa y cómo se procesa e integra dicha información en el estado del grafo.

### **1.1 Flujo de ejecución de la búsqueda web en A002 Refined**

El proceso desde que el agente decide buscar en internet hasta que procesa la respuesta sigue tres etapas clave:

**1. Consulta y Formato de Respuesta**
El modelo formula una consulta a la API de Tavily. La herramienta devuelve una lista de diccionarios JSON (`list[dict]`) donde cada elemento contiene atributos estándar: `title`, `url`, `content` (snippet de texto) y `score` (puntuación de relevancia).

**2. Integración en el Estado (ToolMessage)**
LangGraph recibe la respuesta JSON, la convierte en un texto plano (string serializado) y la inyecta automáticamente en el historial de la conversación empaquetada como un objeto `ToolMessage`. Este mensaje asocia el contenido extraído con el identificador único de la petición (`tool_call_id`).

**3. Impacto en el Consumo de Tokens**
En la versión `A002_refined` no existe ningún filtro por relevancia ni truncado de texto. Todo el payload se vuelca íntegro en el estado, lo que inyecta miles de caracteres de golpe y dispara drásticamente el consumo de tokens de entrada (`prompt_tokens`) en las siguientes iteraciones del agente. Esto se solucionará en `A003_plus_v1` y `A004_plus_v2`

### **1.2 Criterio de Activación de la Herramienta de Búsqueda Web**

Los agentes de esta comparativa (`A002_refined`, `A003_plus_v1` y `A004_plus_v2`) comparten como motor base un modelo cuyo límite de conocimiento estático (*knowledge cutoff*) se sitúa en **julio de 2024**. Por tanto, la activación de la herramienta de búsqueda depende del equilibrio entre el conocimiento preentrenado del modelo y la necesidad de datos actualizados.

**Mecanismo de Decisión del Agente**
* **Uso del conocimiento estático:** Para consultas sobre hechos históricos anteriores a julio de 2024, conceptos teóricos, programación o razonamiento lógico, el agente responde directamente sin recurrir a la red.
* **Activación de Tavily Search:** Ante preguntas sobre eventos posteriores a julio de 2024, datos en tiempo real o información muy específica no garantizada en sus pesos internos, el agente emite un `AIMessage` invocando la herramienta de búsqueda.

**Optimización mediante Prompting**
Aunque los LLMs incorporan una heurística por defecto para determinar cuándo necesitan buscar información externa, es una buena práctica reforzar este criterio explícitamente en el *System Prompt*. De este modo se evita tanto el uso innecesario de la herramienta en consultas simples como la alucinación de datos en eventos recientes.

In [ ]:
# Prompt que no requiere búsqueda en internet
prompt = "¿Cual es el número atómico del Hierro?"
respuesta = agent_refined.invoke({"messages": [HumanMessage(content=prompt)]})

print("=== FLUJO DE CONVERSACIÓN ===")
for msg in respuesta["messages"]:
    msg.pretty_print()

In [ ]:
# Prompt que requiere búsqueda en internet
prompt = "¿Quién ganó los Premios Oscar 2025 en la categoría de Mejor Película y quién obtuvo el premio a Mejor Actor Principal?"
respuesta = agent_refined.invoke({"messages": [HumanMessage(content=prompt)]})

print("=== FLUJO DE CONVERSACIÓN ===")
for msg in respuesta["messages"]:
    msg.pretty_print()

### **1.3 Cálculo y Acumulación de Métricas de Tokens en A002 Refined**

El agente `A002_refined` monitoriza el consumo de recursos apoyándose en dos funciones auxiliares dentro del flujo de LangGraph para registrar el gasto en cada iteración.

**1. Consumo del LLM (`calculate_token_usage`)**
En cada ejecución del nodo `llm_call`, esta función extrae la propiedad `usage_metadata` que devuelve la API de Anthropic y acumula los valores en el estado:
* **Tokens de Prompt (`prompt_tokens`):** Lo que lee el modelo en cada turno (mensaje del sistema, historial y payloads de búsquedas anteriores).
* **Tokens de Respuesta (`completion_tokens`):** Lo que genera el modelo (la respuesta final o los argumentos para llamar a una herramienta).
* **Tokens Totales (`total_tokens`):** La suma acumulada de entrada y salida a lo largo de todas las llamadas al modelo (`llm_calls`).

**2. Estimación en la herramienta (`tool_node`)**
Como la API de Tavily Search no pasa por el modelo y no devuelve tokens oficiales, el nodo `tool_node` calcula una métrica propia llamada `tool_tokens`. Utiliza una fórmula de conversión heurística basada en la densidad media del texto (1 token $\approx$ 4 caracteres):

$$\text{tool\_tokens} = \max\left(1, \left\lfloor \frac{\text{len}(\text{content\_str})}{4} \right\rfloor\right)$$

Esto permite medir el volumen de texto bruto recuperado de internet antes de que sea inyectado en el historial y se convierta en `prompt_tokens` para el siguiente turno del modelo.

**3. Consolidación de métricas (`report_node` y `print_token_and_tool_report`)**
Al finalizar la ejecución, el nodo `report_node` reúne las métricas acumuladas en el estado y genera un mensaje final de tipo `AIMessage` con el resumen consolidado de uso de tokens y herramientas utilizadas.

In [ ]:
# OBSERVAR LAS MÉTRICAS DE TOKENS CON ESTE PROMPT QUE USA TAVILY SEARCH
prompt = "¿Cuáles son los titulares y noticias más destacados de la actualidad internacional en este momento?"
respuesta = agent_refined.invoke({"messages": [HumanMessage(content=prompt)]})

print("=== FLUJO DE CONVERSACIÓN ===")
for msg in respuesta["messages"]:
    msg.pretty_print()

## **2. Evaluación del agente A002 Refined y su search_tool**

En esta sección se evalúa el rendimiento práctico del agente `A002_refined` analizando tres aspectos críticos de su arquitectura: la **calidad de la respuesta generada tras la consulta web**, la **sobrecarga de tokens inyectados al estado** y la **latencia del flujo**.

In [ ]:
# Prompt de evaluación de actualidad
prompt_evaluacion = "¿Cuáles son los titulares y noticias más destacados de la actualidad internacional en este momento?"

# Invocación y medición del agente A002 Refined
resultado_a002 = agent_refined.invoke({"messages": [HumanMessage(content=prompt_evaluacion)]})

# Inspección de los resultados en el estado
print("=== RESPUESTA FINAL DE A002 ===")
print(resultado_a002["messages"][-2].content)  # Respuesta antes del reporte

# Mostrar reporte consolidado de métricas
print(resultado_a002["messages"][-1].content)